In [0]:
import sys
import os

project_root = os.path.abspath('../..')
if project_root not in sys.path:
    sys.path.append(project_root)

from utils.config import *
import datetime

dbutils.widgets.text("data_processamento", datetime.datetime.now().strftime('%Y-%m-%d'))
data_alvo = dbutils.widgets.get("data_processamento")

# Agora usamos a variável vinda do config + o parâmetro do widget
path_today = f"{volume_path}{data_alvo}/"

In [0]:
from pyspark.sql import functions as F

# configurações de checkpoint
checkpoint_base = f"dbfs:/Volumes/{catalog}/{schema}/_checkpoints/bronze/"

# Loop de ingestão
for file in gtfs_files:
    print(f"Iniciando a ingestão do arquivo: {file}")
    
    target_table = f"{catalog}.{schema}.bronze_{file}"
    checkpoint_path = f"{checkpoint_base}{file}"

    # Leitura com Auto Loader
    df_bronze = (spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")  
        .option("cloudFiles.inferColumnTypes", "true")
        .option("pathGlobFilter", f"{file}.txt")
        .option("recursiveFileLookup", "true")
        .option("cloudFiles.schemaLocation", checkpoint_path)
        .load(volume_path))
  
    # Escrita no Unity Catalog
    (df_bronze.writeStream
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(target_table))

    print(f"Ingestão do arquivo {file} concluída com sucesso!")
